In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
image_dir = "/content/drive/MyDrive/ctaA/Preprocessed_Dataset"

folders = os.listdir(image_dir)
print("Files:", folders)

Files: ['.DS_Store', 'kau-bcmd', 'dmid', 'inbreast', 'cdd-cesm', 'cmmd', 'ddsm']


In [5]:
import pandas as pd
csv_path = "/content/drive/MyDrive/ctaA/metadata.csv"

df = pd.read_csv(csv_path)

print(df.head())
print("\nColumns:", df.columns)
print("\nShape:", df.shape)

  source_dataset                       preprocessed_image_path classification  \
0       inbreast  Preprocessed_Dataset/inbreast/inbreast_0.jpg         Normal   
1       inbreast  Preprocessed_Dataset/inbreast/inbreast_1.jpg         Benign   
2       inbreast  Preprocessed_Dataset/inbreast/inbreast_2.jpg         Normal   
3       inbreast  Preprocessed_Dataset/inbreast/inbreast_3.jpg         Benign   
4       inbreast  Preprocessed_Dataset/inbreast/inbreast_4.jpg      Malignant   

  density  BIRADS                      mask_path  \
0       D     1.0  Masks/inbreast/inbreast_0.jpg   
1       D     3.0  Masks/inbreast/inbreast_1.jpg   
2       D     1.0  Masks/inbreast/inbreast_2.jpg   
3       D     3.0  Masks/inbreast/inbreast_3.jpg   
4       B     5.0  Masks/inbreast/inbreast_4.jpg   

                             raw_image_path laterality view source_subjectID  \
0  Original_Dataset/inbreast/inbreast_0.jpg          R   CC         22678622   
1  Original_Dataset/inbreast/inbreast_1.

In [6]:
import os  # lets Python work with folders and file paths

# main folder where extracted images are stored
BASE_PATH = "/content/drive/MyDrive/ctaA"

df["classification"] = df["classification"].replace({
    "Suspicious Malignant": "Malignant"
})

# convert text labels into numbers
label_map = {
    "Normal": 0,
    "Benign": 1,
    "Malignant": 2
}

# make a safe copy of the dataframe
df = df.copy()

# create a new numeric label column from the classification column
df["label"] = df["classification"].map(label_map)

# create the full path to every image
df["full_path"] = df["preprocessed_image_path"].apply(
    lambda x: os.path.join(BASE_PATH, x)
)

# check whether each image file really exists
df["exists"] = df["full_path"].apply(os.path.exists)

# count how many image files are missing
print("Missing images:", (~df["exists"]).sum())

# keep only rows where the image file exists
df = df[df["exists"]].reset_index(drop=True)

# show the final size of the cleaned dataset
print("Final dataset size:", df.shape)

# show first 5 rows
df.head()

Missing images: 0
Final dataset size: (19731, 28)


,source_dataset,preprocessed_image_path,classification,density,BIRADS,mask_path,raw_image_path,laterality,view,source_subjectID,...,abnormality id,calc type,calc distribution,subtlety,mass shape,mass margins,cropped_image_file_new,label,full_path,exists
0,inbreast,Preprocessed_Dataset/inbreast/inbreast_0.jpg,Normal,D,1.0,Masks/inbreast/inbreast_0.jpg,Original_Dataset/inbreast/inbreast_0.jpg,R,CC,22678622,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,/content/drive/MyDrive/ctaA/Preprocessed_Datas...,True
1,inbreast,Preprocessed_Dataset/inbreast/inbreast_1.jpg,Benign,D,3.0,Masks/inbreast/inbreast_1.jpg,Original_Dataset/inbreast/inbreast_1.jpg,L,CC,22678646,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,/content/drive/MyDrive/ctaA/Preprocessed_Datas...,True
2,inbreast,Preprocessed_Dataset/inbreast/inbreast_2.jpg,Normal,D,1.0,Masks/inbreast/inbreast_2.jpg,Original_Dataset/inbreast/inbreast_2.jpg,R,MLO,22678670,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,/content/drive/MyDrive/ctaA/Preprocessed_Datas...,True
3,inbreast,Preprocessed_Dataset/inbreast/inbreast_3.jpg,Benign,D,3.0,Masks/inbreast/inbreast_3.jpg,Original_Dataset/inbreast/inbreast_3.jpg,L,MLO,22678694,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,/content/drive/MyDrive/ctaA/Preprocessed_Datas...,True
4,inbreast,Preprocessed_Dataset/inbreast/inbreast_4.jpg,Malignant,B,5.0,Masks/inbreast/inbreast_4.jpg,Original_Dataset/inbreast/inbreast_4.jpg,R,CC,22614074,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,/content/drive/MyDrive/ctaA/Preprocessed_Datas...,True


In [7]:
from PIL import Image

sample_path = df["full_path"].iloc[10]
img = Image.open(sample_path)

print("Sample path:", sample_path)
print("Image size:", img.size)


Sample path: /content/drive/MyDrive/ctaA/Preprocessed_Dataset/inbreast/inbreast_10.jpg
Image size: (3328, 4084)


In [8]:
print(df["classification"].value_counts())

classification
Malignant    8419
Benign       6069
Normal       5243
Name: count, dtype: int64


In [9]:
from sklearn.model_selection import train_test_split

# Step 1: Create patient-level table
patient_df = df.groupby("source_subjectID").agg({
    "classification": lambda x: x.mode()[0],
    "label": lambda x: x.mode()[0],
    "source_dataset": lambda x: x.mode()[0]
}).reset_index()

# Step 2: Split patients
train_patients, temp_patients = train_test_split(
    patient_df,
    test_size=0.3,
    stratify=patient_df["label"],
    random_state=42
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.5,
    stratify=temp_patients["label"],
    random_state=42
)

# Step 3: Map back to images
train_ids = set(train_patients["source_subjectID"])
val_ids = set(val_patients["source_subjectID"])
test_ids = set(test_patients["source_subjectID"])

train_df = df[df["source_subjectID"].isin(train_ids)].reset_index(drop=True)
val_df = df[df["source_subjectID"].isin(val_ids)].reset_index(drop=True)
test_df = df[df["source_subjectID"].isin(test_ids)].reset_index(drop=True)

# Step 4: Check sizes
print("Train images:", train_df.shape)
print("Validation images:", val_df.shape)
print("Test images:", test_df.shape)

Train images: (13907, 28)
Validation images: (2840, 28)
Test images: (2984, 28)


In [10]:
print(len(set(train_df["source_subjectID"]) & set(val_df["source_subjectID"])))
print(len(set(train_df["source_subjectID"]) & set(test_df["source_subjectID"])))
print(len(set(val_df["source_subjectID"]) & set(test_df["source_subjectID"])))

0
0
0


In [11]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [12]:
from torch.utils.data import Dataset
from PIL import Image

class MammogramDataset(Dataset):
  def __init__(self, dataframe, transform = None):
    self.df = dataframe
    self.transform = transform

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    image_path = self.df.iloc[idx]["full_path"]
    label = self.df.iloc[idx]["label"]

    image = Image.open(image_path).convert("RGB")

    if self.transform:
      image = self.transform(image)

    return image, label

In [13]:
train_dataset = MammogramDataset(train_df, transform=train_transform)
val_dataset = MammogramDataset(val_df, transform=val_test_transform)
test_dataset = MammogramDataset(test_df, transform=val_test_transform)

In [14]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [15]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Labels:", labels[:10])

Image batch shape: torch.Size([32, 3, 224, 224])
Labels: tensor([1, 0, 1, 1, 2, 2, 1, 1, 1, 0])


In [16]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(pretrained=True)

model.fc = nn.Linear(model.fc.in_features, 3)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.0001)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 171MB/s]


In [17]:
epochs = 1
from tqdm.auto import tqdm

for epoch in range(epochs):
  model.train()
  running_loss = 0.0

  for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item()

  print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

Epoch 1/1:   0%|          | 0/435 [00:00<?, ?it/s]

Epoch 1, Loss: 0.7865


In [18]:
torch.save(model.state_dict(), "/content/drive/MyDrive/ctaA/best_model.pth")
print("Model saved successfully.")

Model saved successfully.


In [ ]:
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Accuracy:", accuracy_score(all_labels, all_preds))
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=["Normal", "Benign", "Malignant"]))
print("\nConfusion Matrix:\n")
print(confusion_matrix(all_labels, all_preds))